# Structured output  结构化输出
LangChain 的 create_agent 可以自动处理结构化输出。用户设置所需的结构化输出模式，当模型生成结构化数据时，这些数据会被捕获、验证，然后返回到 agent 状态的 'structured_response' 键中。
## Response format  回复格式
使用 response_format 控制代理返回结构化数据的方式：
- `ToolStrategy[StructuredResponseT]` ：使用工具调用结构化输出
- `ProviderStrategy[StructuredResponseT]` ：使用提供商原生结构化输出
- `type[StructuredResponseT]` ：模式类型 - 根据模型功能自动选择最佳策略
- `None` ：未明确请求结构化输出

结构化响应将以 structured_response 键的形式返回给代理的最终状态。
## 供应商策略
## 工具调用策略:
```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```
**schema  模式 required**

定义结构化输出格式的模式。支持：
- Pydantic 模型 ：带有字段验证的 BaseModel 子类。返回已验证的 Pydantic 实例。
- 数据类 ：带有类型注解的 Python 数据类。返回字典。
- TypedDict ：类型化字典类。返回字典。
- JSON Schema ：包含 JSON 模式规范的字典。返回字典。
- 联合类型 ：多种模式选项。模型将根据上下文选择最合适的模式。

**tool_message_content  工具消息内容**
用于在生成结构化输出时返回工具消息的自定义内容。如果未提供，则默认显示结构化响应数据的消息。

**handle_errors  处理错误**
结构化输出验证失败的错误处理策略。默认为 True 。
- True ：使用默认错误模板捕获所有错误
- str ：使用此自定义消息捕获所有错误
- type[Exception] ：仅捕获此异常类型并使用默认消息
- tuple[type[Exception], ...] ：仅捕获这些异常类型，并显示默认消息
- Callable[[Exception], str] ：返回错误消息的自定义函数
- False ：不重试，允许异常传播

### Custom tool message content 自定义工具消息内容
`tool_message_content` 参数允许您自定义生成结构化输出时显示在对话历史记录中的消息：

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()

base_url = os.getenv("QWEN_BASE_URL")
api_key =  os.getenv("QWEN_API_KEY")

model = ChatOpenAI(
    model="glm-4.7",
    base_url=base_url,
    api_key=api_key,
)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

res = agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})
# for msg in res["messages"]:
#     msg.pretty_print()
res

### Error handling  错误处理
当模型错误地调用多个结构化输出工具时，代理会在 ToolMessage 中提供错误反馈，并提示模型重试：

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Default: handle_errors=True
)

agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

### Schema validation error  架构验证错误
当结构化输出与预期模式不符时，代理会提供具体的错误反馈：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5)
    comment: str = Field(description="Review comment")

agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."
)

res = agent.invoke({
    "messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!remeber,10!"}]
})
for msg in res["messages"]:
    msg.pretty_print()

### Error handling strategies 错误处理策略
可以使用 handle_errors 参数自定义错误处理方式：

**Custom error message** 自定义错误信息：
```python
ToolStrategy(
    schema=ProductRating,
    handle_errors="Please provide a valid rating between 1-5 and include a comment."
)
```
如果 handle_errors 是一个字符串，则代理将始终提示模型使用固定的工具消息重试

**Handle specific exceptions only** 仅处理特定异常情况：
```python
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Only retry on ValueError, raise others
)
```
如果 handle_errors 设置为异常类型，则代理仅在引发的异常类型为指定类型时才会重试（使用默认错误消息）。在所有其他情况下，都会引发异常。

**Handle multiple exception types** 处理多种异常类型：
```python
ToolStrategy(
    schema=ProductRating,
    handle_errors=(ValueError, TypeError)  # Retry on ValueError and TypeError
)
```
如果 handle_errors 是一个异常元组，则代理仅在引发的异常属于指定类型之一时才会重试（使用默认错误消息）。在所有其他情况下，都会引发该异常。

**Custom error handler function** 自定义错误处理函数：

In [ ]:
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"


agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(
                        schema=Union[ContactInfo, EventDetails],
                        handle_errors=custom_error_handler
                    )  # Default: handle_errors=True
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in result['messages']:
    # If message is actually a ToolMessage object (not a dict), check its class name
    if type(msg).__name__ == "ToolMessage":
        print(msg.content)
    # If message is a dictionary or you want a fallback
    elif isinstance(msg, dict) and msg.get('tool_call_id'):
        print(msg['content'])